# Run online STDR mining

This notebook executes the manuscript's next algorithmic steps after network initialization: lazy synapse updates, significant synapse maintenance, and extraction of significant spatio-temporal sequential rules induced by significant directed synapses.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "Source"))

from spatio_temporal_network import (
    LearningParameters,
    OnlineSTDRMiner,
    DependencyRuleSnapshot,
    clear_rule_snapshot_dumps,
    initialize_network_from_events,
)

## Synthetic sanity check

The alternating `A -> B` stream makes it easy to verify that synapse weights become non-zero and rules can be extracted.

In [2]:
synthetic_events = pd.DataFrame(
    {
        "Event_ID": [1, 2, 3, 4, 5, 6],
        "longitude": [0.0, 0.001, 0.0, 0.001, 0.0, 0.001],
        "latitude": [0.0, 0.001, 0.0, 0.001, 0.0, 0.001],
        "timestamp": [0.0, 1.0, 2.0, 3.0, 4.0, 5.0],
        "Event_type": ["A", "B", "A", "B", "A", "B"],
    }
)

synthetic_network = initialize_network_from_events(
    synthetic_events,
    grid_shape=(1, 1),
    spatial_threshold=0.0,
)
synthetic_parameters = LearningParameters(
    eta=0.01,
    lambda_decay=0.001,
    tau_activation=10.0,
    tau_refractory=10.0,
    theta=0.01,
)
synthetic_miner = OnlineSTDRMiner(synthetic_network, synthetic_parameters)
synthetic_rules = synthetic_miner.process_events(synthetic_events)

print(synthetic_network.summary())
print("w_max:", synthetic_parameters.max_weight)
print("significance threshold:", synthetic_parameters.significance_threshold)

{'cells': 1, 'event_types': 2, 'neurons': 2, 'synapses': 2, 'grid_shape': (1, 1), 'spatial_threshold_meters': 0.0}
w_max: 10.0
significance threshold: 0.1


In [3]:
synthetic_miner.significant_synapses_frame()

,presynaptic_id,postsynaptic_id,presynaptic_event_type,postsynaptic_event_type,presynaptic_cell_id,postsynaptic_cell_id,weight,weight_ratio


In [4]:
synthetic_miner.rules_frame(synthetic_rules)

,length,significance,significance_ratio,event_type_rule,presynaptic_id,postsynaptic_id,presynaptic_event_type,postsynaptic_event_type,presynaptic_location,postsynaptic_location,locations,weights,weight_ratios


## Bounded run on TSMC2014 NYC

This example uses timestamp-ordered events from event types selected in the next cell. Keeping the run bounded makes rule extraction fast enough for iterative parameter testing.

In [5]:
csv_path = PROJECT_ROOT / "Data" / "Preprocessed" / "dataset_TSMC2014_NYC_preprocessed.csv"
events = pd.read_csv(csv_path).sort_values("timestamp")

candidate_event_types = events["Event_type"].value_counts().head(20)
candidate_event_types

Event_type
Bar                                         15978
Home (private)                              15382
Office                                      12740
Subway                                       9348
Gym / Fitness Center                         9171
Coffee Shop                                  7510
Food & Drink Shop                            6596
Train Station                                6408
Park                                         4804
Neighborhood                                 4604
Bus Station                                  4474
Deli / Bodega                                4214
Residential Building (Apartment / Condo)     4185
Other Great Outdoors                         4134
American Restaurant                          3701
College Academic Building                    3479
Building                                     3474
Medical Center                               3366
Road                                         3207
Clothing Store                         

In [6]:
# Match the visualization notebook stream: most frequent event types, bounded prefix.
top_event_type_count = 20
max_events = 1000
selected_event_types = events["Event_type"].value_counts().head(top_event_type_count).index.tolist()
selected_events = events[events["Event_type"].isin(selected_event_types)].head(max_events).copy()

missing_event_types = sorted(set(selected_event_types).difference(events["Event_type"].unique()))
if missing_event_types:
    raise ValueError(f"Selected event types are not present in the dataset: {missing_event_types}")
if selected_events.empty:
    raise ValueError("No events found for selected_event_types.")

print("selected event types:", len(selected_event_types))
print("selected events:", len(selected_events))
selected_events["Event_type"].value_counts()


selected event types: 20
selected events: 1000


Event_type
Home (private)                              148
Office                                      100
Bar                                          88
Gym / Fitness Center                         81
Subway                                       74
Coffee Shop                                  57
Train Station                                54
Other Great Outdoors                         47
College Academic Building                    42
Food & Drink Shop                            39
Bus Station                                  37
Road                                         35
Park                                         31
Neighborhood                                 28
Residential Building (Apartment / Condo)     28
Deli / Bodega                                27
Medical Center                               25
Building                                     24
American Restaurant                          22
Clothing Store                               13
Name: count, dtype: int64

In [7]:
from math import log

window_hours = 3
window_minutes = window_hours * 60

# Use the same timestamp windows as Notebooks/qualitative_experiment.ipynb.
event_numbers = []
window_start_indices = []
window_start_index = 0
window_start_time = float(selected_events["timestamp"].iloc[0])

for row_position, timestamp in enumerate(selected_events["timestamp"].to_numpy(), start=1):
    if float(timestamp) - window_start_time >= window_minutes:
        event_numbers.append(row_position)
        window_start_indices.append(window_start_index)
        window_start_index = row_position
        window_start_time = float(timestamp)

if not event_numbers or event_numbers[-1] != len(selected_events):
    event_numbers.append(len(selected_events))
    window_start_indices.append(window_start_index)

checkpoint_start_by_event = dict(zip(event_numbers, window_start_indices))
checkpoint_set = set(event_numbers)

grid_shape = (10, 10)
spatial_threshold = 10_000.0
max_snapshot_rules = None
max_rule_length = None

parameters = LearningParameters(
    eta=1,
    lambda_decay=0.1,#-log(0.10) / window_minutes,
    tau_activation=120,
    tau_refractory=12,
    theta=0.2,
)

network = initialize_network_from_events(
    selected_events,
    grid_shape=grid_shape,
    spatial_threshold=spatial_threshold,
)
miner = OnlineSTDRMiner(network, parameters)

snapshot_dir = PROJECT_ROOT / "Results" / "Tests_online_stdr_mining"
clear_rule_snapshot_dumps(snapshot_dir)

rules = []
dumped_files = []
last_summary = None

for event_index, event in enumerate(selected_events.itertuples(index=False), start=1):
    event_series = pd.Series(event._asdict())
    miner.process_event(event_series, extract_rules=False)
    if event_index not in checkpoint_set:
        continue

    batch_start_index = checkpoint_start_by_event[event_index]
    batch_events = selected_events.iloc[batch_start_index:event_index].copy()
    rules = miner.extract_rules(
        max_rule_length=max_rule_length,
        max_rules=max_snapshot_rules,
    )
    snapshot = DependencyRuleSnapshot(
        processed_events=miner.processed_events,
        event_timestamp=float(event_series["timestamp"]),
        significant_synapse_count=len(miner.significant_synapses),
        rules=tuple(rules),
    )
    dumped_files.append(miner.dump_rule_snapshot(snapshot, snapshot_dir))

    duration_hours = (batch_events["timestamp"].iloc[-1] - batch_events["timestamp"].iloc[0]) / 60.0
    last_summary = {
        "event_number": event_index,
        "batch_start_event": batch_start_index + 1,
        "batch_end_event": event_index,
        "duration_hours": duration_hours,
        "significant_synapses": len(miner.significant_synapses),
        "rules_saved": len(rules),
        "network": network.summary(),
    }
    display(
        Markdown(
            f"### After {event_index} events "
            f"(batch {batch_start_index + 1}-{event_index}, {duration_hours:.2f}h)\n"
            f"Significant synapses: {len(miner.significant_synapses)}; "
            f"saved rules: {len(rules)}"
        )
    )
    display(miner.rules_frame(rules).head(10))

print(network.summary())
print("theta:", parameters.theta)
print("w_max:", parameters.max_weight)
print("weight significance threshold:", parameters.significance_threshold)
print("processed events:", miner.processed_events)
print("significant synapses:", len(miner.significant_synapses))
print("rules:", len(rules))
print("snapshot dump directory:", snapshot_dir)
print("snapshot checkpoints:", len(dumped_files))
print("snapshot files written:", len(dumped_files) * 2)
last_summary


### After 159 events (batch 1-159, 3.00h)
Significant synapses: 271; saved rules: 271

,length,significance,significance_ratio,event_type_rule,presynaptic_id,postsynaptic_id,presynaptic_event_type,postsynaptic_event_type,presynaptic_location,postsynaptic_location,locations,weights,weight_ratios
0,2,7.443089,0.744309,Home (private) -> Residential Building (Apartm...,510,536,Home (private),Residential Building (Apartment / Condo),"(-73.95508605086714, 40.67569738064337)","(-73.89982628039449, 40.67569738064337)","[(-73.95508605086714, 40.67569738064337), (-73...","(7.443088572989973,)","(0.7443088572989973,)"
1,2,7.398731,0.739873,Road -> Residential Building (Apartment / Condo),737,536,Road,Residential Building (Apartment / Condo),"(-73.89982628039449, 40.71432666100203)","(-73.89982628039449, 40.67569738064337)","[(-73.89982628039449, 40.71432666100203), (-73...","(7.398730554356459,)","(0.7398730554356459,)"
2,2,6.636884,0.663688,Subway -> Gym / Fitness Center,98,129,Subway,Gym / Fitness Center,"(-74.01034582133977, 40.598438819926045)","(-73.89982628039449, 40.598438819926045)","[(-74.01034582133977, 40.598438819926045), (-7...","(6.636884397190599,)","(0.6636884397190599,)"
3,2,6.627286,0.662729,Home (private) -> Gym / Fitness Center,510,129,Home (private),Gym / Fitness Center,"(-73.95508605086714, 40.67569738064337)","(-73.89982628039449, 40.598438819926045)","[(-73.95508605086714, 40.67569738064337), (-73...","(6.627286177742386,)","(0.6627286177742386,)"
4,2,6.502855,0.650286,Subway -> Residential Building (Apartment / Co...,738,536,Subway,Residential Building (Apartment / Condo),"(-73.89982628039449, 40.71432666100203)","(-73.89982628039449, 40.67569738064337)","[(-73.89982628039449, 40.71432666100203), (-73...","(6.5028553667440105,)","(0.650285536674401,)"
5,2,6.462816,0.646282,Food & Drink Shop -> Residential Building (Apa...,948,536,Food & Drink Shop,Residential Building (Apartment / Condo),"(-73.84456650992185, 40.752955941360696)","(-73.89982628039449, 40.67569738064337)","[(-73.84456650992185, 40.752955941360696), (-7...","(6.462815768286321,)","(0.6462815768286321,)"
6,2,6.269273,0.626927,Home (private) -> Residential Building (Apartm...,910,536,Home (private),Residential Building (Apartment / Condo),"(-73.95508605086714, 40.752955941360696)","(-73.89982628039449, 40.67569738064337)","[(-73.95508605086714, 40.752955941360696), (-7...","(6.269273226250768,)","(0.6269273226250769,)"
7,2,6.212095,0.621209,Deli / Bodega -> Residential Building (Apartme...,707,536,Deli / Bodega,Residential Building (Apartment / Condo),"(-73.95508605086714, 40.71432666100203)","(-73.89982628039449, 40.67569738064337)","[(-73.95508605086714, 40.71432666100203), (-73...","(6.21209465583297,)","(0.6212094655832969,)"
8,2,6.212095,0.621209,Subway -> Residential Building (Apartment / Co...,718,536,Subway,Residential Building (Apartment / Condo),"(-73.95508605086714, 40.71432666100203)","(-73.89982628039449, 40.67569738064337)","[(-73.95508605086714, 40.71432666100203), (-73...","(6.21209465583297,)","(0.6212094655832969,)"
9,2,6.157806,0.615781,Home (private) -> Gym / Fitness Center,90,129,Home (private),Gym / Fitness Center,"(-74.01034582133977, 40.598438819926045)","(-73.89982628039449, 40.598438819926045)","[(-74.01034582133977, 40.598438819926045), (-7...","(6.157805824831441,)","(0.6157805824831442,)"


### After 391 events (batch 160-391, 2.97h)
Significant synapses: 377; saved rules: 377

,length,significance,significance_ratio,event_type_rule,presynaptic_id,postsynaptic_id,presynaptic_event_type,postsynaptic_event_type,presynaptic_location,postsynaptic_location,locations,weights,weight_ratios
0,2,7.454318,0.745432,Other Great Outdoors -> Bar,694,1101,Other Great Outdoors,Bar,"(-74.01034582133977, 40.71432666100203)","(-73.95508605086714, 40.79158522171935)","[(-74.01034582133977, 40.71432666100203), (-73...","(7.4543178597332975,)","(0.7454317859733297,)"
1,2,7.454104,0.745410,Other Great Outdoors -> Bar,1314,1101,Other Great Outdoors,Bar,"(-73.95508605086714, 40.830214502078015)","(-73.95508605086714, 40.79158522171935)","[(-73.95508605086714, 40.830214502078015), (-7...","(7.454103727027179,)","(0.7454103727027179,)"
2,2,7.449551,0.744955,Home (private) -> Bar,1510,1101,Home (private),Bar,"(-73.95508605086714, 40.86884378243668)","(-73.95508605086714, 40.79158522171935)","[(-73.95508605086714, 40.86884378243668), (-73...","(7.449551252932037,)","(0.7449551252932036,)"
3,2,7.404764,0.740476,Park -> Bar,895,1101,Park,Bar,"(-74.01034582133977, 40.752955941360696)","(-73.95508605086714, 40.79158522171935)","[(-74.01034582133977, 40.752955941360696), (-7...","(7.40476383720903,)","(0.740476383720903,)"
4,2,7.382001,0.738200,Home (private) -> Bar,690,1101,Home (private),Bar,"(-74.01034582133977, 40.71432666100203)","(-73.95508605086714, 40.79158522171935)","[(-74.01034582133977, 40.71432666100203), (-73...","(7.382000658519346,)","(0.7382000658519347,)"
5,2,7.284722,0.728472,American Restaurant -> Bar,900,1101,American Restaurant,Bar,"(-73.95508605086714, 40.752955941360696)","(-73.95508605086714, 40.79158522171935)","[(-73.95508605086714, 40.752955941360696), (-7...","(7.284722280039127,)","(0.7284722280039128,)"
6,2,7.116292,0.711629,Train Station -> Bar,899,1101,Train Station,Bar,"(-74.01034582133977, 40.752955941360696)","(-73.95508605086714, 40.79158522171935)","[(-74.01034582133977, 40.752955941360696), (-7...","(7.1162923222681975,)","(0.7116292322268197,)"
7,2,6.936971,0.693697,Other Great Outdoors -> Neighborhood,694,712,Other Great Outdoors,Neighborhood,"(-74.01034582133977, 40.71432666100203)","(-73.95508605086714, 40.71432666100203)","[(-74.01034582133977, 40.71432666100203), (-73...","(6.936971138611912,)","(0.6936971138611912,)"
8,2,6.916675,0.691667,Bar -> Neighborhood,681,712,Bar,Neighborhood,"(-74.01034582133977, 40.71432666100203)","(-73.95508605086714, 40.71432666100203)","[(-74.01034582133977, 40.71432666100203), (-73...","(6.916674864468571,)","(0.691667486446857,)"
9,2,6.914382,0.691438,Residential Building (Apartment / Condo) -> Ne...,516,712,Residential Building (Apartment / Condo),Neighborhood,"(-73.95508605086714, 40.67569738064337)","(-73.95508605086714, 40.71432666100203)","[(-73.95508605086714, 40.67569738064337), (-73...","(6.914382057460493,)","(0.6914382057460493,)"


### After 567 events (batch 392-567, 3.00h)
Significant synapses: 156; saved rules: 156

...

,length,significance,significance_ratio,event_type_rule,presynaptic_id,postsynaptic_id,presynaptic_event_type,postsynaptic_event_type,presynaptic_location,postsynaptic_location,locations,weights,weight_ratios
0,2,7.400647,0.740065,Gym / Fitness Center -> Home (private),909,510,Gym / Fitness Center,Home (private),"(-73.95508605086714, 40.752955941360696)","(-73.95508605086714, 40.67569738064337)","[(-73.95508605086714, 40.752955941360696), (-7...","(7.400646773000636,)","(0.7400646773000636,)"
1,2,7.395607,0.739561,Subway -> Home (private),118,510,Subway,Home (private),"(-73.95508605086714, 40.598438819926045)","(-73.95508605086714, 40.67569738064337)","[(-73.95508605086714, 40.598438819926045), (-7...","(7.395607270459247,)","(0.7395607270459247,)"
2,2,7.388741,0.738874,Residential Building (Apartment / Condo) -> Ho...,716,510,Residential Building (Apartment / Condo),Home (private),"(-73.95508605086714, 40.71432666100203)","(-73.95508605086714, 40.67569738064337)","[(-73.95508605086714, 40.71432666100203), (-73...","(7.388740704782438,)","(0.7388740704782438,)"
3,2,7.326927,0.732693,Deli / Bodega -> Home (private),687,510,Deli / Bodega,Home (private),"(-74.01034582133977, 40.71432666100203)","(-73.95508605086714, 40.67569738064337)","[(-74.01034582133977, 40.71432666100203), (-73...","(7.326927030663664,)","(0.7326927030663664,)"
4,2,7.145507,0.714551,Office -> Home (private),913,510,Office,Home (private),"(-73.95508605086714, 40.752955941360696)","(-73.95508605086714, 40.67569738064337)","[(-73.95508605086714, 40.752955941360696), (-7...","(7.145506883951358,)","(0.7145506883951358,)"
5,2,7.001214,0.700121,Deli / Bodega -> Home (private),887,510,Deli / Bodega,Home (private),"(-74.01034582133977, 40.752955941360696)","(-73.95508605086714, 40.67569738064337)","[(-74.01034582133977, 40.752955941360696), (-7...","(7.001214128372253,)","(0.7001214128372253,)"
6,2,6.907939,0.690794,American Restaurant -> Home (private),900,510,American Restaurant,Home (private),"(-73.95508605086714, 40.752955941360696)","(-73.95508605086714, 40.67569738064337)","[(-73.95508605086714, 40.752955941360696), (-7...","(6.907938866748925,)","(0.6907938866748925,)"
7,2,6.892335,0.689233,Gym / Fitness Center -> Home (private),889,510,Gym / Fitness Center,Home (private),"(-74.01034582133977, 40.752955941360696)","(-73.95508605086714, 40.67569738064337)","[(-74.01034582133977, 40.752955941360696), (-7...","(6.892334775345826,)","(0.6892334775345825,)"
8,2,6.880766,0.688077,Coffee Shop -> Home (private),885,510,Coffee Shop,Home (private),"(-74.01034582133977, 40.752955941360696)","(-73.95508605086714, 40.67569738064337)","[(-74.01034582133977, 40.752955941360696), (-7...","(6.880765955540643,)","(0.6880765955540643,)"
9,2,6.876568,0.687657,Building -> Home (private),902,510,Building,Home (private),"(-73.95508605086714, 40.752955941360696)","(-73.95508605086714, 40.67569738064337)","[(-73.95508605086714, 40.752955941360696), (-7...","(6.876567518980751,)","(0.6876567518980752,)"


### After 1000 events (batch 850-1000, 1.38h)
Significant synapses: 405; saved rules: 405

,length,significance,significance_ratio,event_type_rule,presynaptic_id,postsynaptic_id,presynaptic_event_type,postsynaptic_event_type,presynaptic_location,postsynaptic_location,locations,weights,weight_ratios
0,2,7.443950,0.744395,Office -> Medical Center,933,911,Office,Medical Center,"(-73.89982628039449, 40.752955941360696)","(-73.95508605086714, 40.752955941360696)","[(-73.89982628039449, 40.752955941360696), (-7...","(7.443949769686372,)","(0.7443949769686372,)"
1,2,7.439795,0.743979,Food & Drink Shop -> Medical Center,728,911,Food & Drink Shop,Medical Center,"(-73.89982628039449, 40.71432666100203)","(-73.95508605086714, 40.752955941360696)","[(-73.89982628039449, 40.71432666100203), (-73...","(7.439794983733938,)","(0.7439794983733938,)"
2,2,7.354919,0.735492,Subway -> Medical Center,918,911,Subway,Medical Center,"(-73.95508605086714, 40.752955941360696)","(-73.95508605086714, 40.752955941360696)","[(-73.95508605086714, 40.752955941360696), (-7...","(7.354919155991391,)","(0.7354919155991391,)"
3,2,7.350036,0.735004,American Restaurant -> Medical Center,680,911,American Restaurant,Medical Center,"(-74.01034582133977, 40.71432666100203)","(-73.95508605086714, 40.752955941360696)","[(-74.01034582133977, 40.71432666100203), (-73...","(7.350035790578394,)","(0.7350035790578394,)"
4,2,7.343619,0.734362,Food & Drink Shop -> Medical Center,1108,911,Food & Drink Shop,Medical Center,"(-73.95508605086714, 40.79158522171935)","(-73.95508605086714, 40.752955941360696)","[(-73.95508605086714, 40.79158522171935), (-73...","(7.343619395044537,)","(0.7343619395044537,)"
5,2,7.341962,0.734196,Deli / Bodega -> Medical Center,907,911,Deli / Bodega,Medical Center,"(-73.95508605086714, 40.752955941360696)","(-73.95508605086714, 40.752955941360696)","[(-73.95508605086714, 40.752955941360696), (-7...","(7.341962063529971,)","(0.7341962063529971,)"
6,2,7.318927,0.731893,Office -> Medical Center,913,911,Office,Medical Center,"(-73.95508605086714, 40.752955941360696)","(-73.95508605086714, 40.752955941360696)","[(-73.95508605086714, 40.752955941360696), (-7...","(7.318927023885843,)","(0.7318927023885843,)"
7,2,7.277551,0.727755,Park -> Medical Center,895,911,Park,Medical Center,"(-74.01034582133977, 40.752955941360696)","(-73.95508605086714, 40.752955941360696)","[(-74.01034582133977, 40.752955941360696), (-7...","(7.277551339034179,)","(0.727755133903418,)"
8,2,7.247848,0.724785,Gym / Fitness Center -> Medical Center,909,911,Gym / Fitness Center,Medical Center,"(-73.95508605086714, 40.752955941360696)","(-73.95508605086714, 40.752955941360696)","[(-73.95508605086714, 40.752955941360696), (-7...","(7.247847582483632,)","(0.7247847582483632,)"
9,2,7.241627,0.724163,Deli / Bodega -> Medical Center,687,911,Deli / Bodega,Medical Center,"(-74.01034582133977, 40.71432666100203)","(-73.95508605086714, 40.752955941360696)","[(-74.01034582133977, 40.71432666100203), (-73...","(7.2416272351340885,)","(0.7241627235134088,)"


{'cells': 100, 'event_types': 20, 'neurons': 2000, 'synapses': 528960, 'grid_shape': (10, 10), 'spatial_threshold_meters': 10000.0}
theta: 0.2
w_max: 10.0
weight significance threshold: 2.0
processed events: 1000
significant synapses: 405
rules: 405
snapshot dump directory: /Users/piotr/Praca/Nauka/publikacje/SNNs_ST_Patterns/Experiments_software/Results/Tests_online_stdr_mining
snapshot checkpoints: 7
snapshot files written: 14


{'event_number': 1000,
 'batch_start_event': 850,
 'batch_end_event': 1000,
 'duration_hours': np.float64(1.378611111111108),
 'significant_synapses': 405,
 'rules_saved': 405,
 'network': {'cells': 100,
  'event_types': 20,
  'neurons': 2000,
  'synapses': 528960,
  'grid_shape': (10, 10),
  'spatial_threshold_meters': 10000.0}}

In [8]:
miner.significant_synapses_frame().head(20)

,presynaptic_id,postsynaptic_id,presynaptic_event_type,postsynaptic_event_type,presynaptic_cell_id,postsynaptic_cell_id,weight,weight_ratio
0,933,911,Office,Medical Center,46,45,7.443950,0.744395
1,728,911,Food & Drink Shop,Medical Center,36,45,7.439795,0.743979
2,918,911,Subway,Medical Center,45,45,7.354919,0.735492
3,680,911,American Restaurant,Medical Center,34,45,7.350036,0.735004
4,1108,911,Food & Drink Shop,Medical Center,55,45,7.343619,0.734362
5,907,911,Deli / Bodega,Medical Center,45,45,7.341962,0.734196
6,913,911,Office,Medical Center,45,45,7.318927,0.731893
7,895,911,Park,Medical Center,44,45,7.277551,0.727755
8,909,911,Gym / Fitness Center,Medical Center,45,45,7.247848,0.724785
9,687,911,Deli / Bodega,Medical Center,34,45,7.241627,0.724163


In [9]:
miner.rules_frame(rules).head(20)

,length,significance,significance_ratio,event_type_rule,presynaptic_id,postsynaptic_id,presynaptic_event_type,postsynaptic_event_type,presynaptic_location,postsynaptic_location,locations,weights,weight_ratios
0,2,7.443950,0.744395,Office -> Medical Center,933,911,Office,Medical Center,"(-73.89982628039449, 40.752955941360696)","(-73.95508605086714, 40.752955941360696)","[(-73.89982628039449, 40.752955941360696), (-7...","(7.443949769686372,)","(0.7443949769686372,)"
1,2,7.439795,0.743979,Food & Drink Shop -> Medical Center,728,911,Food & Drink Shop,Medical Center,"(-73.89982628039449, 40.71432666100203)","(-73.95508605086714, 40.752955941360696)","[(-73.89982628039449, 40.71432666100203), (-73...","(7.439794983733938,)","(0.7439794983733938,)"
2,2,7.354919,0.735492,Subway -> Medical Center,918,911,Subway,Medical Center,"(-73.95508605086714, 40.752955941360696)","(-73.95508605086714, 40.752955941360696)","[(-73.95508605086714, 40.752955941360696), (-7...","(7.354919155991391,)","(0.7354919155991391,)"
3,2,7.350036,0.735004,American Restaurant -> Medical Center,680,911,American Restaurant,Medical Center,"(-74.01034582133977, 40.71432666100203)","(-73.95508605086714, 40.752955941360696)","[(-74.01034582133977, 40.71432666100203), (-73...","(7.350035790578394,)","(0.7350035790578394,)"
4,2,7.343619,0.734362,Food & Drink Shop -> Medical Center,1108,911,Food & Drink Shop,Medical Center,"(-73.95508605086714, 40.79158522171935)","(-73.95508605086714, 40.752955941360696)","[(-73.95508605086714, 40.79158522171935), (-73...","(7.343619395044537,)","(0.7343619395044537,)"
5,2,7.341962,0.734196,Deli / Bodega -> Medical Center,907,911,Deli / Bodega,Medical Center,"(-73.95508605086714, 40.752955941360696)","(-73.95508605086714, 40.752955941360696)","[(-73.95508605086714, 40.752955941360696), (-7...","(7.341962063529971,)","(0.7341962063529971,)"
6,2,7.318927,0.731893,Office -> Medical Center,913,911,Office,Medical Center,"(-73.95508605086714, 40.752955941360696)","(-73.95508605086714, 40.752955941360696)","[(-73.95508605086714, 40.752955941360696), (-7...","(7.318927023885843,)","(0.7318927023885843,)"
7,2,7.277551,0.727755,Park -> Medical Center,895,911,Park,Medical Center,"(-74.01034582133977, 40.752955941360696)","(-73.95508605086714, 40.752955941360696)","[(-74.01034582133977, 40.752955941360696), (-7...","(7.277551339034179,)","(0.727755133903418,)"
8,2,7.247848,0.724785,Gym / Fitness Center -> Medical Center,909,911,Gym / Fitness Center,Medical Center,"(-73.95508605086714, 40.752955941360696)","(-73.95508605086714, 40.752955941360696)","[(-73.95508605086714, 40.752955941360696), (-7...","(7.247847582483632,)","(0.7247847582483632,)"
9,2,7.241627,0.724163,Deli / Bodega -> Medical Center,687,911,Deli / Bodega,Medical Center,"(-74.01034582133977, 40.71432666100203)","(-73.95508605086714, 40.752955941360696)","[(-74.01034582133977, 40.71432666100203), (-73...","(7.2416272351340885,)","(0.7241627235134088,)"
